# AIPROD — Générer votre Première Vidéo sur Google Colab Pro

**GPU requis :** A100 40GB (Colab Pro)  
**Durée totale :** ~10 minutes (téléchargement modèle) + ~5 minutes (génération)  
**Coût :** Inclus dans Colab Pro (~10$/mois)  

---

### Ce que fait ce notebook

| Étape | Action | Durée |
|-------|--------|-------|
| 1 | Vérifier GPU A100 | 5 sec |
| 2 | Cloner le repo AIPROD | 30 sec |
| 2b | Monter Google Drive (cache modèles + vidéos) | 10 sec |
| 3 | Installer les dépendances | 2-3 min |
| 4 | Authentification HuggingFace | 10 sec |
| 5 | **Générer une vidéo + audio** (sauvegarde auto sur Drive) | 10-15 min |
| 6 | Télécharger / visualiser le résultat | 10 sec |

> **Résultat :** Un fichier `.mp4` avec vidéo + audio générés par IA via le pipeline **propriétaire AIPROD**.  
> **Google Drive :** Les modèles (~20 GB) sont mis en cache → les sessions suivantes démarrent en **~30 sec** !

---

### Prérequis

1. **Google Colab Pro** (~10$/mois) — [colab.research.google.com](https://colab.research.google.com)
2. Sélectionner **Runtime > Change runtime type > A100 GPU**
3. Un **token HuggingFace** (gratuit) — [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
4. Accepter la licence **Gemma 3** — [huggingface.co/google/gemma-3-1b-pt](https://huggingface.co/google/gemma-3-1b-pt)

## Étape 1 — Vérifier le GPU

Exécutez cette cellule pour confirmer que vous avez bien un **A100**.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ Pas de GPU détecté !\n"
        "   → Allez dans Runtime > Change runtime type > GPU (A100)\n"
        "   → Si A100 n'apparaît pas, vérifiez votre abonnement Colab Pro"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
vram_free = torch.cuda.mem_get_info(0)[0] / 1024**3

print(f"✅ GPU détecté : {gpu_name}")
print(f"   VRAM : {vram_total:.0f} GB total, {vram_free:.0f} GB libre")
print(f"   PyTorch : {torch.__version__}")
print(f"   CUDA : {torch.version.cuda}")

if vram_total < 20:
    print()
    print("⚠️  ATTENTION : Vous avez un GPU avec < 20 GB VRAM.")
    print("   Le modèle 19B FP8 nécessite ~20 GB minimum.")
    print("   → Changez pour un A100 : Runtime > Change runtime type > A100")
elif vram_total >= 35:
    print()
    print("🎉 Parfait ! A100 40GB détecté — idéal pour AIPROD.")
else:
    print()
    print("✅ GPU suffisant pour la génération vidéo.")

## Étape 2 — Cloner le repo AIPROD

Clone le code source (~7 MB). Les modèles seront téléchargés séparément.

In [ ]:
import os

REPO_URL = "https://github.com/Blockprod/AIPROD.git"
WORK_DIR = "/content/AIPROD"

# Si repo privé, décommentez et ajoutez votre token GitHub :
# REPO_URL = "https://<VOTRE_TOKEN_GITHUB>@github.com/Blockprod/AIPROD.git"

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull origin main

%cd {WORK_DIR}
print(f"\n✅ Repo AIPROD prêt dans {os.getcwd()}")

## Étape 3 — Installer les dépendances

Installe PyTorch + le pipeline propriétaire **AIPROD** (~2-3 minutes).

In [ ]:
# Installer PyTorch avec CUDA 12.1
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Backend de diffusion (dépendance interne AIPROD)
%pip install -q git+https://github.com/huggingface/diffusers

# Dépendances essentielles
%pip install -q transformers accelerate safetensors huggingface_hub
%pip install -q sentencepiece protobuf  # Tokenizer Gemma 3
%pip install -q pillow imageio rich pyyaml tqdm einops
%pip install -q av  # PyAV pour encodage MP4

# ── Installer les packages AIPROD propriétaires ──
import sys, os

AIPROD_CANDIDATES = [
    os.getcwd(),
    "/content/AIPROD",
    os.path.expanduser("~/AIPROD"),
]

AIPROD_ROOT = None
for candidate in AIPROD_CANDIDATES:
    if os.path.isdir(os.path.join(candidate, "packages", "aiprod-core", "src")):
        AIPROD_ROOT = candidate
        break

if AIPROD_ROOT is None:
    print("❌ Le repo AIPROD n'a pas été trouvé !")
    print("   → Exécutez d'abord la cellule 2 (Cloner le repo)")
    raise FileNotFoundError("Exécutez la cellule 2 avant celle-ci !")

print(f"📂 AIPROD trouvé dans : {AIPROD_ROOT}")
os.chdir(AIPROD_ROOT)

# Ajouter les packages propriétaires AIPROD au path
for pkg in ["packages/aiprod-core/src", "packages/aiprod-pipelines/src"]:
    pkg_path = os.path.join(AIPROD_ROOT, pkg)
    if pkg_path not in sys.path:
        sys.path.insert(0, pkg_path)
    print(f"   ✅ {pkg}")

# Purger les anciens modules AIPROD du cache (force rechargement après git pull)
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("aiprod_core", "aiprod_pipelines")):
        del sys.modules[mod_name]

# Vérifier l'installation
import aiprod_core
import aiprod_pipelines
print(f"\n✅ Installation terminée")
print(f"   aiprod-core : OK")
print(f"   aiprod-pipelines : OK")
print(f"   AIPRODVideoGenerator : disponible")
print(f"   torch : {__import__('torch').__version__}")

## Étape 4 — Authentification HuggingFace

Requis pour télécharger le text encoder **Gemma 3** (modèle gated par Google).

> **Comment obtenir un token :**
> 1. Créez un compte sur [huggingface.co](https://huggingface.co)
> 2. Acceptez la licence Gemma : [google/gemma-3-1b-pt](https://huggingface.co/google/gemma-3-1b-pt)
> 3. Créez un token : [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (type **Read**)
> 4. Dans Colab : panneau gauche 🔑 > **Add secret** > Name: `HF_TOKEN`, Value: votre token

In [ ]:
from huggingface_hub import login

# ═══════════════════════════════════════════════════════════════
# Authentification HuggingFace (requis pour Gemma 3)
# ═══════════════════════════════════════════════════════════════
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("✅ Authentifié sur HuggingFace via Colab Secret")
except Exception:
    hf_token = None
    print("⚠️  Pas de HF_TOKEN trouvé dans les Secrets Colab.")
    print()
    print("   Pour obtenir un token :")
    print("   1. Allez sur https://huggingface.co/google/gemma-3-1b-pt → Accepter la licence")
    print("   2. Créez un token : https://huggingface.co/settings/tokens")
    print("   3. Panneau gauche 🔑 > Add secret > Name: HF_TOKEN")
    print("   4. Relancez cette cellule")
    raise PermissionError("HF_TOKEN requis pour télécharger Gemma 3")

## Étape 5 — GÉNÉRER VOTRE VIDÉO ! 🎬

Utilise le pipeline propriétaire **`AIPRODVideoGenerator`** pour générer vidéo + audio.

**Modifiez le `PROMPT` ci-dessous**, puis exécutez la cellule.

| Paramètre | Valeur par défaut | Description |
|-----------|-------------------|-------------|
| `PROMPT` | *Un drone survolant...* | Votre description de la vidéo |
| `NUM_FRAMES` | 121 | Nombre d'images (~5 sec à 24fps) |
| `STEPS` | 40 | Qualité (plus = meilleur mais plus lent) |
| `HEIGHT × WIDTH` | 512 × 768 | Résolution |
| `GUIDANCE_SCALE` | 4.0 | Fidélité au prompt (3-7) |

> **Premier lancement :** le modèle (~20 GB) sera téléchargé automatiquement.  
> **Durée estimée :** 10-15 min (download) + 3-5 min (génération).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5. GÉNÉRATION VIDÉO + AUDIO 🎬 (PC local, sans Google Drive)
# ═══════════════════════════════════════════════════════════════

import os, time, shutil, torch
from aiprod_pipelines.generate import AIPRODVideoGenerator, GenerationConfig

# ── Forcer le cache et les modèles sur le disque local ──────────
# IMPORTANT : Ceci évite toute utilisation de Google Drive pour le cache/models
os.environ.pop('HF_HOME', None)
os.environ.pop('TRANSFORMERS_CACHE', None)
os.environ.pop('HF_DATASETS_CACHE', None)
os.environ['HF_HOME'] = os.path.abspath('AIPROD/models_cache')
os.environ['TRANSFORMERS_CACHE'] = os.path.abspath('AIPROD/models_cache')
os.environ['HF_DATASETS_CACHE'] = os.path.abspath('AIPROD/models_cache')

# ── Paramètres de génération ───────────────────────────────────
PROMPT = (
    "A cinematic slow-motion drone shot over misty mountains at golden hour, "
    "volumetric god rays piercing through dramatic clouds, "
    "camera slowly pulling back to reveal a vast valley below, "
    "professional cinematography, 4K, shallow depth of field"
)
WIDTH = 768
HEIGHT = 512
NUM_FRAMES = 121        # ~5 secondes à 24 fps
STEPS = 40              # qualité optimale
GUIDANCE_SCALE = 4.0
SEED = 42
OUTPUT_PATH = "output/aiprod_video.mp4"  # Chemin local sur PC

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ── Résumé ─────────────────────────────────────────────────────
print("═" * 60)
print("🎬 AIPROD — Génération Vidéo + Audio (PC local)")
print("═" * 60)
print(f"   Prompt   : {PROMPT[:80]}..." if len(PROMPT) > 80 else f"   Prompt   : {PROMPT}")
print(f"   Résolution : {HEIGHT}×{WIDTH}")
print(f"   Frames   : {NUM_FRAMES} (~{NUM_FRAMES/24:.1f}s à 24fps)")
print(f"   Steps    : {STEPS}")
print(f"   Guidance : {GUIDANCE_SCALE}")
print(f"   Seed     : {SEED}")
print(f"   Output   : {OUTPUT_PATH}")
print()

# ── Initialiser le générateur AIPROD ───────────────────────────
print("⏳ Initialisation du générateur AIPROD...")
t_start = time.time()

# Variante automatique : on ne passe PAS le paramètre variant (compatibilité maximale)
generator = AIPRODVideoGenerator(
    device="cuda:0",
    dtype=torch.bfloat16,
    cpu_offload=True,
    enable_tiling=True,
    local_files_only=False,
    # PAS de paramètre variant ici !
    # Si besoin d'économiser la RAM, essayez dtype=torch.float16
    # Pour un modèle custom, ajoutez model_id ou cache_dir
    # Ex: model_id="AIPROD/models/aiprod-shdt-v1-bf16.safetensors"
    # Ex: cache_dir="AIPROD/models_cache"
    # Ex: local_files_only=True pour forcer le cache uniquement
)

# ── Générer ────────────────────────────────────────────────────
print(f"🎨 Génération en cours ({STEPS} étapes de diffusion)...")

video, audio = generator.generate(
    prompt=PROMPT,
    output_path=OUTPUT_PATH,
    config=GenerationConfig(
        prompt=PROMPT,
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE_SCALE,
        seed=SEED,
    ),
)

total_time = time.time() - t_start

# Libérer la VRAM
generator.release()
del generator
torch.cuda.empty_cache()

# ── Résultat ──────────────────────────────────────────────────
from pathlib import Path
mp4 = Path(OUTPUT_PATH)
size_mb = mp4.stat().st_size / 1024**2
duration = NUM_FRAMES / 24.0

print()
print("═" * 60)
print("🎉 VIDÉO GÉNÉRÉE AVEC SUCCÈS !")
print("═" * 60)
print(f"   📁 Fichier  : {OUTPUT_PATH}")
print(f"   💾 Taille   : {size_mb:.1f} MB")
print(f"   ⏱️  Durée    : {duration:.1f} secondes")
print(f"   📐 Résolution : {HEIGHT}×{WIDTH}")
print(f"   🔊 Audio    : Oui (synchronisé)")
print(f"   ⏱️  Temps    : {total_time:.0f} secondes")

print()
print("▶️ Exécutez la cellule suivante pour visualiser la vidéo ⬇️")

## Étape 6 — Télécharger votre vidéo

La vidéo est **automatiquement sauvegardée sur Google Drive** (`MyDrive/AIPROD/videos/`).  
Utilisez les options ci-dessous pour la télécharger directement dans votre navigateur.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTION A : Téléchargement direct (navigateur)
# ═══════════════════════════════════════════════════════════════
from google.colab import files  # type: ignore[import-not-found]
files.download(OUTPUT_PATH)
print(f"📥 Téléchargement lancé : {OUTPUT_PATH}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTION B : Vérifier les vidéos sur Google Drive
# ═══════════════════════════════════════════════════════════════
import os

DRIVE_VIDEOS = "/content/drive/MyDrive/AIPROD/videos"
if os.path.isdir(DRIVE_VIDEOS):
    videos = [f for f in os.listdir(DRIVE_VIDEOS) if f.endswith(".mp4")]
    print(f"📂 {len(videos)} vidéo(s) sur Google Drive ({DRIVE_VIDEOS}) :")
    for v in sorted(videos):
        size = os.path.getsize(f"{DRIVE_VIDEOS}/{v}") / 1024**2
        print(f"   📹 {v} ({size:.1f} MB)")
else:
    print("⚠️ Google Drive non monté — exécutez l'étape 2b")

## Étape 7 — Visualiser la vidéo dans Colab

Affiche la vidéo directement dans le notebook.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

with open(OUTPUT_PATH, "rb") as f:
    video_data = f.read()

b64 = b64encode(video_data).decode("ascii")
HTML(f"""
<h3>🎬 Résultat : {os.path.basename(OUTPUT_PATH)}</h3>
<video width="768" controls autoplay loop>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
<p><b>Prompt :</b> {PROMPT}</p>
""")

---

## Bonus — Générer plusieurs vidéos

Modifiez les prompts et lancez cette cellule pour générer un lot de vidéos.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GÉNÉRATION EN LOT — Modifiez les prompts ci-dessous
# ═══════════════════════════════════════════════════════════════

BATCH_PROMPTS = [
    "A peaceful zen garden with cherry blossoms falling in slow motion",
    "A futuristic city at night with neon lights and flying cars",
    "Ocean waves crashing on a tropical beach at sunset, cinematic",
]

BATCH_STEPS = 40
BATCH_FRAMES = 121
BATCH_HEIGHT = 512
BATCH_WIDTH = 768
BATCH_GUIDANCE = 4.0

# ═══════════════════════════════════════════════════════════════

import os, re, time, shutil, torch
os.chdir("/content/AIPROD")

from aiprod_pipelines.generate import AIPRODVideoGenerator, GenerationConfig

# Charger le générateur AIPROD une seule fois
print("⏳ Initialisation du générateur AIPROD...")
generator = AIPRODVideoGenerator(
    device="cuda:0",
    dtype=torch.bfloat16,
    cpu_offload=True,
    enable_tiling=True,
    local_files_only=False,
    variant="bf16",
)

DRIVE_VIDEOS = "/content/drive/MyDrive/AIPROD/videos"
drive_ok = os.path.isdir("/content/drive/MyDrive")

for i, prompt in enumerate(BATCH_PROMPTS, 1):
    safe_name = re.sub(r"[^a-zA-Z0-9_ ]", "", prompt)[:50].strip().replace(" ", "_")
    out_path = f"/content/output/{safe_name}.mp4"
    
    print(f"\n{'═'*60}")
    print(f"🎬 Vidéo {i}/{len(BATCH_PROMPTS)} : {prompt[:60]}")
    print(f"{'═'*60}")
    
    t0 = time.time()
    video, audio = generator.generate(
        prompt=prompt,
        output_path=out_path,
        config=GenerationConfig(
            prompt=prompt,
            width=BATCH_WIDTH,
            height=BATCH_HEIGHT,
            num_frames=BATCH_FRAMES,
            num_inference_steps=BATCH_STEPS,
            guidance_scale=BATCH_GUIDANCE,
            seed=42 + i,
        ),
    )
    
    size_mb = os.path.getsize(out_path) / 1024**2
    elapsed = time.time() - t0
    print(f"✅ Sauvegardé : {out_path} ({size_mb:.1f} MB, {elapsed:.0f}s)")
    
    # Auto-save sur Google Drive
    if drive_ok:
        os.makedirs(DRIVE_VIDEOS, exist_ok=True)
        shutil.copy2(out_path, f"{DRIVE_VIDEOS}/{safe_name}.mp4")
        print(f"   ☁️ → Google Drive")
    
    del video, audio
    torch.cuda.empty_cache()

# Libérer le générateur
generator.release()
del generator
torch.cuda.empty_cache()

print(f"\n{'═'*60}")
print(f"🎉 {len(BATCH_PROMPTS)} vidéos générées !")
print(f"{'═'*60}")
print(f"   📂 Local : /content/output/")
if drive_ok:
    print(f"   ☁️  Drive : {DRIVE_VIDEOS}")

for f in sorted(os.listdir("/content/output")):
    if f.endswith(".mp4"):
        size = os.path.getsize(f"/content/output/{f}") / 1024**2
        print(f"   📹 {f} ({size:.1f} MB)")

---

## Bonus — Génération avec LoRA AIPROD (après entraînement)

Si vous avez entraîné le modèle LoRA SHDT (notebook d'entraînement),  
vous pouvez l'appliquer ici pour des résultats personnalisés.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GÉNÉRATION AVEC LORA AIPROD (optionnel — après entraînement)
# ═══════════════════════════════════════════════════════════════
# Quand votre modèle SHDT propriétaire sera entraîné, vous pourrez
# l'utiliser directement via AIPRODVideoGenerator en changeant le
# model_id vers votre checkpoint local :
#
# from aiprod_pipelines.generate import AIPRODVideoGenerator
#
# generator = AIPRODVideoGenerator(
#     model_id="/content/drive/MyDrive/AIPROD/trained_models/aiprod-shdt-v1",
# )
# video, audio = generator.generate(prompt="Your prompt here")
#
# Pour le moment, le backend utilise LTX-2 comme modèle de base.
# L'API AIPROD reste identique — seul le backend change.

print("ℹ️ Le générateur AIPROD supporte les modèles personnalisés.")
print("   Quand votre modèle SHDT sera entraîné, changez simplement le model_id.")
print("   L'API propriétaire reste la même.")

---

## FAQ & Dépannage

### ❌ `CUDA out of memory`
- Assurez-vous d'avoir un **A100** (pas T4 ni V100)
- Réduisez `NUM_FRAMES` à 61 ou `HEIGHT/WIDTH` à 384×512
- Le pipeline utilise `cpu_offload=True` pour limiter la VRAM

### ❌ `401 Client Error: Unauthorized`
- Vérifiez votre `HF_TOKEN` dans les Secrets Colab (🔑)
- Acceptez la licence Gemma 3 : [huggingface.co/google/gemma-3-1b-pt](https://huggingface.co/google/gemma-3-1b-pt)

### ❌ `No module named 'aiprod_pipelines'`
- Relancez la cellule 3 (installation des packages)
- Vérifiez que le repo AIPROD a été cloné (cellule 2)

### ❌ Vidéo toute noire / bruit
- Essayez un `SEED` différent (ex: 42, 123, 7)
- Augmentez `STEPS` à 50
- Augmentez `GUIDANCE_SCALE` à 5.0 ou 6.0
- Utilisez un prompt plus descriptif en anglais

### 💡 Astuces pour de bons prompts
- Utilisez l'**anglais** pour de meilleurs résultats
- Soyez **descriptif** : mouvement de caméra, éclairage, style
- Ajoutez des mots clés : `cinematic`, `4K`, `slow motion`, `aerial shot`
- Exemples :
  - `"A slow-motion close-up of rain drops falling on a flower petal"`
  - `"Aerial shot of a mountain lake at sunrise, mist rising, 4K cinematic"`
  - `"A cat sitting on a windowsill watching snowfall, cozy room, warm light"`

### 🔊 Audio
- Le modèle génère **vidéo + audio synchronisés**
- L'audio est automatiquement inclus dans le MP4
- Décrivez l'ambiance sonore dans le prompt pour de meilleurs résultats

### 🛡️ Architecture AIPROD
- Le notebook utilise **uniquement** l'API propriétaire `aiprod_pipelines`
- Le backend de diffusion est un détail d'implémentation interne
- L'architecture SHDT sera disponible quand le modèle propriétaire sera entraîné

### 💰 Coût estimé
- Colab Pro : ~10$/mois = **illimité** (dans les limites GPU)
- ~5 min par vidéo → **~50-100 vidéos par session**
- Pas de frais supplémentaires par vidéo

---

# Utilisation optimale de l'espace disque local Colab

**Recommandations pour stocker et gérer intelligemment vos fichiers sur Colab :**

1. **Ne montez pas Google Drive**
   - Ignorez toute cellule qui monte Google Drive.

2. **Stockez tout dans `/content`**
   - Modèles : `/content/AIPROD/models_cache`
   - Vidéos générées : `/content/output`

3. **Configurez le cache HuggingFace sur le disque local**
   - Ajoutez cette cellule au début du notebook :

```python
import os
os.environ['HF_HOME'] = '/content/AIPROD/models_cache'
os.environ['TRANSFORMERS_CACHE'] = '/content/AIPROD/models_cache'
os.environ['HF_DATASETS_CACHE'] = '/content/AIPROD/models_cache'
```

4. **Surveillez l'espace disque**
   - Pour voir l'espace restant :
     ```python
     !df -h /
     ```
   - Pour lister les plus gros dossiers/fichiers :
     ```python
     !du -h /content | sort -hr | head -20
     ```

5. **Nettoyez régulièrement**
   - Supprimez les fichiers inutiles pour libérer de l'espace :
     ```python
     import shutil
     shutil.rmtree('/content/output')  # supprime tous les outputs
     ```

6. **Attention à la persistance**
   - Tout ce qui est dans `/content` sera effacé à la fin de la session Colab.

**Workflow optimal :**
- Clonez le repo dans `/content/AIPROD`
- Téléchargez les modèles dans `/content/AIPROD/models_cache`
- Générez les vidéos dans `/content/output`
- Surveillez et nettoyez l'espace disque régulièrement

Vous utilisez ainsi 100 % l'espace local Colab, sans dépendre de Google Drive, et évitez les saturations disque.